Import necessary modules and functions

In [2]:
pip install python-dotenv langchain langchain-community langchain-openai google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client


In [1]:
import os
from dotenv import load_dotenv
from langchain_community.agent_toolkits import GmailToolkit
from langchain_community.tools.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)
from langchain import hub
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain_openai import ChatOpenAI

Section 1: Load Environment Variables

In [2]:
# Load environment variables from .env file
load_dotenv()

# Ensure the OPENAI_API_KEY environment variable is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("The OPENAI_API_KEY environment variable is not set")

Section 2: Set Up Gmail Credentials and Toolkit

In [5]:
# Set up Gmail credentials
credentials = get_gmail_credentials(
    token_file="token.json",
    scopes=["https://mail.google.com/"],
    client_secrets_file="credentials.json",
)

# Build API resource service for Gmail
api_resource = build_resource_service(credentials=credentials)

# Initialize Gmail toolkit with the API resource
toolkit = GmailToolkit(api_resource=api_resource)
tools = toolkit.get_tools()

Section 3: Configure OpenAI API Key

In [6]:
# Use the environment variable for the API key
os.environ["OPENAI_API_KEY"] = api_key

Section 4: Create Prompt and Language Model

In [7]:
# Define instructions for the assistant
instructions = """You are an assistant. my name is Lainitha. I want you to help me with my emails.
You can use the tools provided to read, send, and manage my emails.
when you are writing emails put signature as "lainitha , team Decryptogen". and also put the subject as 'Regarding your email', """

# Pull the base prompt template
base_prompt = hub.pull("langchain-ai/openai-functions-template")

# Customize the base prompt with specific instructions
prompt = base_prompt.partial(instructions=instructions)

# Initialize the language model with specified temperature
llm = ChatOpenAI(temperature=0)

c:\Users\lainitha\anaconda3\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Section 5: Create and Configure the Agent

In [9]:
# Create an agent using OpenAI functions and the Gmail toolkit
agent = create_openai_functions_agent(llm, toolkit.get_tools(), prompt)

# Initialize the AgentExecutor with the agent and tools
agent_executor = AgentExecutor(
    agent=agent,
    tools=toolkit.get_tools(),
    verbose=False,  # Set to False to prevent sensitive email information from being displayed
)

Section 6: Invoke the Agent to Send an Email

In [12]:
# Define the input command to send an email
input_command = {
    "input": " draft an email to lainithac1@gmail.com "
}

# Invoke the agent with the input command
agent_executor.invoke(input_command)

{'input': ' draft an email to lainithac1@gmail.com ',
 'output': 'I have drafted an email for you. Would you like to review it before sending?'}